# British Empire Knowledge Graph — Verified Fixes

Compiled from 6 history expert verification agents (March 2026).  
Each cell is self-contained and can be run independently.

**Neo4j**: `bolt://206.12.90.118:7687` — set `NEO4J_PASSWORD` env var or enter when prompted.

## Fix Categories
1. Setup & connection
2. Pre-fix verification (counts)
3. Delete duplicate/spurious nodes
4. Fix dates (established_year, independence_year)
5. Fix names, types, and QIDs
6. Delete bogus edges
7. Fix edge types
8. Add new edge type: ADMINISTERED_UNDER
9. Add missing nodes
10. Add missing edges
11. Region assignments
12. Post-fix verification
13. Re-export empire_viz_data.json
14. Historian review additions

In [16]:
# Cell 1: Setup & Connection
from neo4j import GraphDatabase
import json
import os
import getpass

URI = os.environ.get("NEO4J_URI", "bolt://206.12.90.118:7687")
USER = os.environ.get("NEO4J_USER", "neo4j")
PASSWORD = os.environ.get("NEO4J_PASSWORD") or getpass.getpass("Neo4j password: ")

driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))

def run(query, **params):
    """Run a single Cypher query and return results as list of dicts."""
    with driver.session() as s:
        result = s.run(query, **params)
        return [dict(r) for r in result]

def run_write(query, **params):
    """Run a write query and return summary counters."""
    with driver.session() as s:
        result = s.run(query, **params)
        summary = result.consume()
        c = summary.counters
        parts = []
        if c.nodes_created: parts.append(f"nodes_created={c.nodes_created}")
        if c.nodes_deleted: parts.append(f"nodes_deleted={c.nodes_deleted}")
        if c.relationships_created: parts.append(f"rels_created={c.relationships_created}")
        if c.relationships_deleted: parts.append(f"rels_deleted={c.relationships_deleted}")
        if c.properties_set: parts.append(f"props_set={c.properties_set}")
        return ", ".join(parts) if parts else "no changes"

# Test connection
result = run("MATCH (n:Colony) RETURN count(n) as cnt")
print(f"Connected. Colony nodes: {result[0]['cnt']}")

Neo4j password:  ········


Connected. Colony nodes: 276


In [8]:
# Cell 2: Pre-fix verification — snapshot current state
print("=== PRE-FIX STATE ===")
for label in ['Colony', 'Territory', 'Place']:
    r = run(f"MATCH (n:{label}) RETURN count(n) as cnt")
    print(f"  {label}: {r[0]['cnt']}")

r = run("MATCH ()-[r]->() WHERE type(r) IN ['EVOLVED_INTO','PARTITIONED_INTO','PART_OF'] RETURN type(r) as t, count(r) as cnt ORDER BY t")
print("\nEdge counts:")
for row in r:
    print(f"  {row['t']}: {row['cnt']}")

# Check for duplicates we plan to delete
dupes = [
    'canada_french_1608_1763', 'australia_1901_1942',
    'uganda_protectorate_1894_1962', 'somaliland_protectorate_1884_1960',
    'anglo_egyptian_sudan_1899_1956', 'james_island_gambia_1661_1779',
    'southern_rhodesia_1901_1980', 'northern_rhodesia_1911_1964',
    'mandatory_palestine_1920_1948', 'kuwait_protectorate_1899_1961',
    'british_hong_kong_1842_1997', 'singapore_1824_1965', 'malaya_1824_1963',
    'indian_empire_(british_raj)_1613_1947', 'east_india_company_pre_territorial_1612_1757',
    'fort_st_george_madras_1640_1857', 'british_guyana_1831_1966',
    'newfoundland_1497_1949', 'louisiana_french_1682_1762',
    'manitoba_1870_1870_ongoing', 'northwest_territories_1870_1870_ongoing',
    'northern_territory_1911_ongoing', 'australian_capital_territory_1911_ongoing',
    'northern_rhodesia_post_federation_1963',
    'nova_scotia_scottish_1621_1632',
    'macao_british_co_administration_1622_1623',
]
r = run("""
    UNWIND $ids AS id
    OPTIONAL MATCH (n:Colony {colony_id: id})
    RETURN id, n IS NOT NULL as exists
""", ids=dupes)
existing = [row['id'] for row in r if row['exists']]
print(f"\nDuplicate/spurious nodes found in KG: {len(existing)} of {len(dupes)}")
for nid in existing:
    print(f"  - {nid}")

=== PRE-FIX STATE ===
  Colony: 299
  Territory: 1
  Place: 6236431

Edge counts:
  EVOLVED_INTO: 183
  PARTITIONED_INTO: 2
  PART_OF: 200895

Duplicate/spurious nodes found in KG: 23 of 26
  - canada_french_1608_1763
  - uganda_protectorate_1894_1962
  - somaliland_protectorate_1884_1960
  - anglo_egyptian_sudan_1899_1956
  - james_island_gambia_1661_1779
  - southern_rhodesia_1901_1980
  - northern_rhodesia_1911_1964
  - mandatory_palestine_1920_1948
  - kuwait_protectorate_1899_1961
  - british_hong_kong_1842_1997
  - singapore_1824_1965
  - malaya_1824_1963
  - east_india_company_pre_territorial_1612_1757
  - fort_st_george_madras_1640_1857
  - british_guyana_1831_1966
  - newfoundland_1497_1949
  - louisiana_french_1682_1762
  - manitoba_1870_1870_ongoing
  - northwest_territories_1870_1870_ongoing
  - northern_territory_1911_ongoing
  - australian_capital_territory_1911_ongoing
  - northern_rhodesia_post_federation_1963
  - macao_british_co_administration_1622_1623


In [9]:
# Cell 3: Delete duplicate/spurious nodes
# Each node is DETACH DELETEd (removes all edges too).
# Edges that should be redirected to canonical nodes are handled in Cell 11.

DELETE_IDS = [
    # Exact duplicates (same entity, same QID, same dates)
    ('canada_french_1608_1763', 'duplicate of New France (same dates)'),
    ('australia_1901_1942', 'duplicate of Commonwealth of Australia (same QID Q408)'),
    ('uganda_protectorate_1894_1962', 'duplicate of Uganda (Q1097943)'),
    ('somaliland_protectorate_1884_1960', 'duplicate of British Somaliland (Q662653)'),
    ('anglo_egyptian_sudan_1899_1956', 'duplicate of Sudan, Anglo-Egyptian (Q541455)'),
    ('james_island_gambia_1661_1779', 'duplicate of James Island Trading Post (shorter dates)'),
    ('mandatory_palestine_1920_1948', 'duplicate of Palestine (Q193714)'),
    ('kuwait_protectorate_1899_1961', 'duplicate of Kuwait (Q3480281)'),
    ('british_hong_kong_1842_1997', 'duplicate of Hong Kong (Q1054923)'),
    ('british_guyana_1831_1966', 'duplicate of British Guiana (wrong spelling, wrong est)'),
    
    # Envelope bars (overlapping with sub-entities, cause visual clutter)
    ('southern_rhodesia_1901_1980', 'envelope bar; lineage goes through SR Colony (1923-1953)'),
    ('northern_rhodesia_1911_1964', 'envelope bar; lineage goes through NR Colony (1924-1953)'),
    ('singapore_1824_1965', 'envelope bar overlapping Settlement + Crown Colony'),
    ('malaya_1824_1963', 'envelope bar overlapping Straits Settlements + Malayan Union + Federation'),
    ('indian_empire_(british_raj)_1613_1947', 'envelope bar, wrong est (1613), overlaps presidencies'),
    ('east_india_company_pre_territorial_1612_1757', 'teleological framing, not a real admin unit'),
    ('northern_rhodesia_post_federation_1963', 'replaced by _1964 version to break graph cycle'),
    
    # Duplicates of existing nodes (same entity, different representation)
    ('fort_st_george_madras_1640_1857', 'Fort St. George IS Madras Presidency (same entity, same year 1640)'),
    
    # Not British colonies
    ('newfoundland_1497_1949', '1497 is Cabots arrival, not a colony; Colony of Newfoundland (1610) is the real start'),
    ('louisiana_french_1682_1762', 'French colony, never became British'),
    ('nova_scotia_scottish_1621_1632', 'no effective colonial presence; paper charter only'),
    ('macao_british_co_administration_1622_1623', 'spurious entity; 1622 was a failed Dutch invasion of Portuguese Macao'),
    
    # Internal subdivisions (not separate British colonies)
    ('manitoba_1870_1870_ongoing', 'internal Canadian subdivision'),
    ('northwest_territories_1870_1870_ongoing', 'internal Canadian subdivision'),
    ('northern_territory_1911_ongoing', 'internal Australian territory'),
    ('australian_capital_territory_1911_ongoing', 'internal Australian territory'),
]

for nid, reason in DELETE_IDS:
    result = run_write("MATCH (n:Colony {colony_id: $id}) DETACH DELETE n", id=nid)
    if 'no changes' in result:
        print(f"  SKIP (not found): {nid}")
    else:
        print(f"  DELETED: {nid} — {reason} [{result}]")

print(f"\nProcessed {len(DELETE_IDS)} deletion targets.")

  DELETED: canada_french_1608_1763 — duplicate of New France (same dates) [nodes_deleted=1, rels_deleted=3]
  SKIP (not found): australia_1901_1942
  DELETED: uganda_protectorate_1894_1962 — duplicate of Uganda (Q1097943) [nodes_deleted=1, rels_deleted=4]
  DELETED: somaliland_protectorate_1884_1960 — duplicate of British Somaliland (Q662653) [nodes_deleted=1, rels_deleted=1]
  DELETED: anglo_egyptian_sudan_1899_1956 — duplicate of Sudan, Anglo-Egyptian (Q541455) [nodes_deleted=1, rels_deleted=2]
  DELETED: james_island_gambia_1661_1779 — duplicate of James Island Trading Post (shorter dates) [nodes_deleted=1, rels_deleted=1]
  DELETED: mandatory_palestine_1920_1948 — duplicate of Palestine (Q193714) [nodes_deleted=1, rels_deleted=2]
  DELETED: kuwait_protectorate_1899_1961 — duplicate of Kuwait (Q3480281) [nodes_deleted=1, rels_deleted=3]
  DELETED: british_hong_kong_1842_1997 — duplicate of Hong Kong (Q1054923) [nodes_deleted=1, rels_deleted=1]
  DELETED: british_guyana_1831_1966 — d

In [10]:
# Cell 4: Fix dates (established_year, independence_year)
# Format: (node_id, field, new_value, reason)

DATE_FIXES = [
    # Forts/factories: end when successor Presidency is established
    ('factory_at_surat_1612_1857', 'independence_year', 1687, 'EIC moved HQ to Bombay 1687'),
    ('fort_william_calcutta_1696_1857', 'independence_year', 1757, 'Battle of Plassey → Bengal Presidency'),
    
    # Australia/NZ: Dominion ended at Statute of Westminster adoption
    ('commonwealth_of_australia_1901_ongoing', 'independence_year', 1942, 'Statute of Westminster Adoption Act 1942'),
    
    # British Columbia merger
    ('mainland_british_columbia_1858_1866', 'independence_year', 1866, 'merged with Vancouver Island 1866'),
    ('united_colony_of_bc_1866_1871', 'independence_year', 1871, 'joined Confederation 1871'),
    
    # Newfoundland: Colony became Dominion in 1907
    ('colony_of_newfoundland_1610_1949', 'independence_year', 1907, 'became Dominion 1907'),
    
    # New Hampshire: 1629 is John Masons province grant
    ('new_hampshire_colony_1623_1783', 'established_year', 1629, 'province grant 1629, not 1623 fishing settlement'),
    
    # Canada
    ('canada_dominion_of_1867_ongoing', 'independence_year', 1931, 'Statute of Westminster 1931'),
    
    # Caribbean
    ('leeward_islands_colony_1671_1816', 'independence_year', 1816, 'reorganized 1816, not 1956'),
    ('federal_colony_leeward_islands_1833-1960', 'independence_year', 1960, 'dissolved 1960'),
    ('antigua_colony_1632_1981', 'independence_year', 1981, 'independence as Antigua and Barbuda'),
    ('trinidad_and_tobago_colony_1797_1962', 'independence_year', 1889, 'united with Tobago 1 Jan 1889'),
    ('settlement_of_belize_1798_1862', 'independence_year', 1862, 'became British Honduras'),
    
    # Acadia end date
    ('acadia_french_1604_1713', 'independence_year', 1713, 'Treaty of Utrecht'),
    
    # South Asia
    ('assam_province_1874_1905', 'independence_year', 1905, 'merged into Eastern Bengal and Assam'),
    ('lower_burma_1824_1886', 'independence_year', 1886, 'absorbed into Burma (British India)'),
    ('assam_bengal_presidency_1826_1874', 'independence_year', 1874, 'separated as own province'),
    ('assam_province_restored_1912_1947', 'independence_year', 1947, 'Indian independence'),
    
    # Aden
    ('aden_1839_1963', 'independence_year', 1937, 'became Crown Colony 1937'),
    
    # Southeast Asia
    ('bencoolen_bengkulu_1685_1824', 'independence_year', 1824, 'ceded to Netherlands, Anglo-Dutch Treaty'),
    ('banda_islands_british_occupation_1810_1817', 'independence_year', 1817, 'returned to Netherlands'),
    
    # Rhodesia/Nyasaland
    ('nyasaland_1891_1964', 'independence_year', 1953, 'joined Federation; post-federation node covers 1963-1964'),
    
    # Sierra Leone
    ('sierra_leone_colony_and_protectorate_1787_1961', 'established_year', 1896, 'protectorate declared 1896, not 1787 Province of Freedom'),
    
    # Lagos Colony fix
    ('lagos_protectorate_1887_1906', 'established_year', 1862, 'Lagos Colony established 1862, not 1887'),
]

for nid, field, value, reason in DATE_FIXES:
    result = run_write(f"MATCH (n:Colony {{colony_id: $id}}) SET n.{field} = $val", id=nid, val=value)
    status = "OK" if "props_set" in result else "SKIP (not found)"
    print(f"  {status}: {nid}.{field} = {value} — {reason}")

print(f"\nProcessed {len(DATE_FIXES)} date fixes.")

  OK: factory_at_surat_1612_1857.independence_year = 1687 — EIC moved HQ to Bombay 1687
  OK: fort_william_calcutta_1696_1857.independence_year = 1757 — Battle of Plassey → Bengal Presidency
  OK: commonwealth_of_australia_1901_ongoing.independence_year = 1942 — Statute of Westminster Adoption Act 1942
  OK: mainland_british_columbia_1858_1866.independence_year = 1866 — merged with Vancouver Island 1866
  OK: united_colony_of_bc_1866_1871.independence_year = 1871 — joined Confederation 1871
  OK: colony_of_newfoundland_1610_1949.independence_year = 1907 — became Dominion 1907
  OK: new_hampshire_colony_1623_1783.established_year = 1629 — province grant 1629, not 1623 fishing settlement
  OK: canada_dominion_of_1867_ongoing.independence_year = 1931 — Statute of Westminster 1931
  OK: leeward_islands_colony_1671_1816.independence_year = 1816 — reorganized 1816, not 1956
  OK: federal_colony_leeward_islands_1833-1960.independence_year = 1960 — dissolved 1960
  OK: antigua_colony_1632_1981

In [11]:
# Cell 5: Fix names, types, and QIDs

NAME_TYPE_QID_FIXES = [
    # Name fixes
    ('canada_dominion_of_1867_ongoing', {'canonical_name': 'Dominion of Canada'}),
    ('trinidad_and_tobago_colony_1797_1962', {'canonical_name': 'Trinidad Colony'}),
    ('trinidad_and_tobago_1889_1962', {'canonical_name': 'Colony of Trinidad and Tobago'}),
    ('new_south_wales_original_1788_1901', {'canonical_name': 'New South Wales'}),
    ('lagos_protectorate_1887_1906', {'canonical_name': 'Lagos Colony'}),
    
    # Type fixes
    ('federal_colony_leeward_islands_1833-1960', {'type': 'Federal Colony'}),
    ('assam_province_restored_1912_1947', {'type': 'Province'}),
    ('nyasaland_post_federation_1963', {'type': 'Transitional', 'established_year': 1963, 'independence_year': 1964}),
    
    # QID fixes — verified by history agents
    ('zululand_1887_1897', {'wikidata_id': 'Q729768'}),  # Zulu Kingdom (user-provided)
    ('newfoundland_commission_1934_1949', {'wikidata_id': 'Q5152678'}),  # was Q48378 ("pulley"!)
    ('st_vincent_colony_1763_1979', {'wikidata_id': 'Q15240384'}),  # was Q757 (modern nation)
    ('henderson_island_1902_ongoing', {'wikidata_id': 'Q184851'}),  # was Q35672 (Pitcairn Islands)
    ('james_island_trading_post_1661_1816', {'wikidata_id': 'Q1254836'}),  # was blank
]

for nid, props in NAME_TYPE_QID_FIXES:
    set_clauses = ", ".join(f"n.{k} = ${k}" for k in props)
    result = run_write(f"MATCH (n:Colony {{colony_id: $id}}) SET {set_clauses}", id=nid, **props)
    status = "OK" if "props_set" in result else "SKIP (not found)"
    print(f"  {status}: {nid} — {props}")

print(f"\nProcessed {len(NAME_TYPE_QID_FIXES)} name/type/QID fixes.")

  OK: canada_dominion_of_1867_ongoing — {'canonical_name': 'Dominion of Canada'}
  OK: trinidad_and_tobago_colony_1797_1962 — {'canonical_name': 'Trinidad Colony'}
  OK: trinidad_and_tobago_1889_1962 — {'canonical_name': 'Colony of Trinidad and Tobago'}
  OK: new_south_wales_original_1788_1901 — {'canonical_name': 'New South Wales'}
  OK: lagos_protectorate_1887_1906 — {'canonical_name': 'Lagos Colony'}
  OK: federal_colony_leeward_islands_1833-1960 — {'type': 'Federal Colony'}
  OK: assam_province_restored_1912_1947 — {'type': 'Province'}
  OK: nyasaland_post_federation_1963 — {'type': 'Transitional', 'established_year': 1963, 'independence_year': 1964}
  OK: zululand_1887_1897 — {'wikidata_id': 'Q729768'}
  SKIP (not found): newfoundland_commission_1934_1949 — {'wikidata_id': 'Q5152678'}
  OK: st_vincent_colony_1763_1979 — {'wikidata_id': 'Q15240384'}
  OK: henderson_island_1902_ongoing — {'wikidata_id': 'Q184851'}
  OK: james_island_trading_post_1661_1816 — {'wikidata_id': 'Q1254836

In [12]:
# Cell 6: Delete bogus edges
# Format: (source_id, target_id, reason)

DELETE_EDGES = [
    # Cape Colony didn't partition into these — ORS was proclaimed over trans-Orange territory; Natalia was a Boer republic
    ('cape_colony_british_1806_1910', 'natalia_republic_1839_1843', 'Cape annexed Natalia, not partitioned into it'),
    ('cape_colony_british_1806_1910', 'orange_river_sovereignty_1848_1854', 'ORS was not Cape territory'),
    
    # Federation → Rhodesia UDI: should go through post-federation SR node
    ('federation_of_rhodesia_and_nyasaland_1953_1963', 'rhodesia_udi_1965_1979', 'should flow through post-fed SR'),
    
    # Tanganyika → Zanzibar: Zanzibar predates Tanganyika! Independent lineages that merge in 1964
    ('tanganyika_territory_1922_1961', 'zanzibar_1890_1963', 'Zanzibar (1890) predates Tanganyika (1922)'),
    
    # SA → Northern Territory: federal admin transfer, not colonial succession
    ('south_australia_1836_1901', 'northern_territory_1911_ongoing', 'federal admin transfer'),
    
    # WIF membership, not succession
    ('west_indies_federation_1958_1962', 'trinidad_and_tobago_1889_1962', 'WIF membership, not succession'),
    ('west_indies_federation_1958_1962', 'jamaica_1655_1962', 'WIF membership, not succession; Jamaica existed 307 years before WIF'),
    
    # Concurrent parallel protectorates, not succession
    ('southern_nigeria_protectorate_1900_1914', 'northern_nigeria_1900_1914', 'concurrent parallel protectorates'),
    
    # Wrong Assam node
    ('assam_province_1874_1905', 'dominion_of_india_1947_1950', 'wrong Assam; restored version has this edge'),
    ('assam_province_restored_1912_1947', 'dominion_of_pakistan_1947_1956', 'Assam went to India, not Pakistan'),
    
    # BC: concurrent colonies that merged, not succession
    ('vancouver_island_1849_1866', 'mainland_british_columbia_1858_1866', 'concurrent colonies that merged'),
    ('mainland_british_columbia_1858_1866', 'canada_dominion_of_1867_ongoing', 'United Colony joined, not Mainland'),
    
    # Gold Coast: Ashanti and Togoland were not Gold Coast territory
    # (will be replaced with ADMINISTERED_UNDER in Cell 8)
    ('gold_coast_colony_1874_1957', 'ashanti_1901_1957', 'Ashanti was conquered Ashanti Empire territory, not GC partition'),
    ('gold_coast_colony_1874_1957', 'british_togoland_1919_1957', 'Togoland came from German Togoland, not Gold Coast'),
    
    # Fiji <-> Western Australia bogus geographic edge
    ('fiji_1874_1970', 'western_australia_1832_1901', 'bogus geographic edge'),
    
    # Duplicate edges (second copy)
    ('lagos_protectorate_1887_1906', 'southern_nigeria_protectorate_1900_1914', 'duplicate edge (keep one)'),
    ('southern_nigeria_protectorate_1900_1914', 'colony_and_protectorate_of_nigeria_1914_1960', 'duplicate edge (keep one)'),
]

for src, tgt, reason in DELETE_EDGES:
    # Delete all relationship types between the pair
    result = run_write("""
        MATCH (s:Colony {colony_id: $src})-[r]->(t:Colony {colony_id: $tgt})
        DELETE r
    """, src=src, tgt=tgt)
    status = "OK" if "rels_deleted" in result else "SKIP (not found)"
    print(f"  {status}: {src} -> {tgt} — {reason}")

# For duplicate edges, keep one copy (delete only extras)
for src, tgt in [
    ('lagos_protectorate_1887_1906', 'southern_nigeria_protectorate_1900_1914'),
    ('southern_nigeria_protectorate_1900_1914', 'colony_and_protectorate_of_nigeria_1914_1960'),
]:
    # Re-add one clean edge
    run_write("""
        MATCH (s:Colony {colony_id: $src}), (t:Colony {colony_id: $tgt})
        MERGE (s)-[:EVOLVED_INTO]->(t)
    """, src=src, tgt=tgt)

print(f"\nProcessed {len(DELETE_EDGES)} edge deletions.")

  OK: cape_colony_british_1806_1910 -> natalia_republic_1839_1843 — Cape annexed Natalia, not partitioned into it
  OK: cape_colony_british_1806_1910 -> orange_river_sovereignty_1848_1854 — ORS was not Cape territory
  OK: federation_of_rhodesia_and_nyasaland_1953_1963 -> rhodesia_udi_1965_1979 — should flow through post-fed SR
  OK: tanganyika_territory_1922_1961 -> zanzibar_1890_1963 — Zanzibar (1890) predates Tanganyika (1922)
  SKIP (not found): south_australia_1836_1901 -> northern_territory_1911_ongoing — federal admin transfer
  OK: west_indies_federation_1958_1962 -> trinidad_and_tobago_1889_1962 — WIF membership, not succession
  OK: west_indies_federation_1958_1962 -> jamaica_1655_1962 — WIF membership, not succession; Jamaica existed 307 years before WIF
  OK: southern_nigeria_protectorate_1900_1914 -> northern_nigeria_1900_1914 — concurrent parallel protectorates
  OK: assam_province_1874_1905 -> dominion_of_india_1947_1950 — wrong Assam; restored version has this edge
  OK

In [13]:
# Cell 7: Fix edge types (DELETE old + CREATE correct type)
# Format: (source_id, target_id, old_type, new_type, reason)

EDGE_TYPE_FIXES = [
    # BSAC Territory was SPLIT between SR and NR (Pattern E: Partition)
    ('british_south_africa_company_territory_1889_1923', 'southern_rhodesia_colony_1923_1953',
     'EVOLVED_INTO', 'PARTITIONED_INTO', 'territory divided between SR + NR'),
    ('british_south_africa_company_territory_1889_1923', 'northern_rhodesia_colony_1924_1953',
     'EVOLVED_INTO', 'PARTITIONED_INTO', 'territory divided between SR + NR'),
    
    # Leeward Islands Colony partition into sub-groupings
    ('leeward_islands_colony_1671_1816', 'antigua_montserrat_barbuda_1816_1833',
     'EVOLVED_INTO', 'PARTITIONED_INTO', 'colony split into two sub-groupings'),
    ('leeward_islands_colony_1671_1816', 'stkitts_nevis_anguilla_1816_1833',
     'EVOLVED_INTO', 'PARTITIONED_INTO', 'colony split into two sub-groupings'),
    
    # Bengal Crown continued after carve-outs — PARTITIONED_INTO, not EVOLVED_INTO
    ('bengal_presidency_crown_1858_1912', 'assam_province_1874_1905',
     'EVOLVED_INTO', 'PARTITIONED_INTO', 'Assam carved out while Bengal continued'),
    ('bengal_presidency_crown_1858_1912', 'eastern_bengal_and_assam_1905_1912',
     'EVOLVED_INTO', 'PARTITIONED_INTO', 'Bengal divided 1905; parent continued as Bengal Partitioned'),
    ('bengal_presidency_crown_1858_1912', 'bengal_province_partitioned_1905_1912',
     'EVOLVED_INTO', 'PARTITIONED_INTO', 'Bengal divided 1905; this is the reduced Bengal'),
    ('bengal_presidency_crown_1858_1912', 'bihar_and_orissa_1912_1947',
     'EVOLVED_INTO', 'PARTITIONED_INTO', 'Bihar & Orissa carved out while Bengal continued'),
    
    # Punjab → NWFP carve-out
    ('punjab_province_1849_1947', 'north_west_frontier_province_1901_1947',
     'EVOLVED_INTO', 'PARTITIONED_INTO', 'NWFP carved from Punjab; Punjab continued'),
    
    # 1947 Partition: Bengal and Punjab divided between India and Pakistan
    ('bengal_province_reunited_1912_1947', 'dominion_of_india_1947_1950',
     'EVOLVED_INTO', 'PARTITIONED_INTO', 'Bengal divided at partition'),
    ('bengal_province_reunited_1912_1947', 'dominion_of_pakistan_1947_1956',
     'EVOLVED_INTO', 'PARTITIONED_INTO', 'Bengal divided at partition'),
    ('punjab_province_1849_1947', 'dominion_of_india_1947_1950',
     'EVOLVED_INTO', 'PARTITIONED_INTO', 'Punjab divided at partition'),
    ('punjab_province_1849_1947', 'dominion_of_pakistan_1947_1956',
     'EVOLVED_INTO', 'PARTITIONED_INTO', 'Punjab divided at partition'),
    
    # NSW carve-outs (NSW continued after each)
    ('new_south_wales_original_1788_1901', 'van_diemens_land_1803_1856',
     'EVOLVED_INTO', 'PARTITIONED_INTO', 'VDL carved from NSW; NSW continued'),
    ('new_south_wales_original_1788_1901', 'victoria_colony_1851_1901',
     'EVOLVED_INTO', 'PARTITIONED_INTO', 'Victoria carved from NSW; NSW continued'),
    ('new_south_wales_original_1788_1901', 'queensland_1859_1901',
     'EVOLVED_INTO', 'PARTITIONED_INTO', 'Queensland carved from NSW; NSW continued'),
    
    # Straits Settlements → Singapore Crown Colony (Straits was partitioned, not just evolved)
    ('straits_settlements_1826_1946', 'singapore_crown_colony_1946_1963',
     'EVOLVED_INTO', 'PARTITIONED_INTO', 'Straits dissolved 1946; peninsular → Malayan Union, Singapore → CC'),
    
    # Straits Settlements → Malayan Union (also a partition, not evolution)
    ('straits_settlements_1826_1946', 'malayan_union_1946_1948',
     'EVOLVED_INTO', 'PARTITIONED_INTO', 'Straits partitioned among multiple successors'),
    
    # Gilbert and Ellice Islands split into two (partition, not evolution)
    ('gilbert_and_ellice_islands_1916_1976', 'gilbert_islands_1976_1979',
     'EVOLVED_INTO', 'PARTITIONED_INTO', 'colony divided into two successors'),
    ('gilbert_and_ellice_islands_1916_1976', 'ellice_islands_1976_1978',
     'EVOLVED_INTO', 'PARTITIONED_INTO', 'colony divided into two successors'),
    
    # Royal Niger Company territory was divided between N and S Nigeria
    ('royal_niger_company_territory_1886_1900', 'northern_nigeria_1900_1914',
     'EVOLVED_INTO', 'PARTITIONED_INTO', 'RNC territory divided between N + S Nigeria'),
]

for src, tgt, old_type, new_type, reason in EDGE_TYPE_FIXES:
    # Delete old edge
    result1 = run_write(f"""
        MATCH (s:Colony {{colony_id: $src}})-[r:{old_type}]->(t:Colony {{colony_id: $tgt}})
        DELETE r
    """, src=src, tgt=tgt)
    # Create new edge
    result2 = run_write(f"""
        MATCH (s:Colony {{colony_id: $src}}), (t:Colony {{colony_id: $tgt}})
        MERGE (s)-[:{new_type}]->(t)
    """, src=src, tgt=tgt)
    d = "rels_deleted" in result1
    c = "rels_created" in result2
    print(f"  {'OK' if d and c else 'CHECK'}: {src} -> {tgt}: {old_type} → {new_type} — {reason}")

print(f"\nProcessed {len(EDGE_TYPE_FIXES)} edge type fixes.")

  OK: british_south_africa_company_territory_1889_1923 -> southern_rhodesia_colony_1923_1953: EVOLVED_INTO → PARTITIONED_INTO — territory divided between SR + NR
  CHECK: british_south_africa_company_territory_1889_1923 -> northern_rhodesia_colony_1924_1953: EVOLVED_INTO → PARTITIONED_INTO — territory divided between SR + NR
  OK: leeward_islands_colony_1671_1816 -> antigua_montserrat_barbuda_1816_1833: EVOLVED_INTO → PARTITIONED_INTO — colony split into two sub-groupings
  OK: leeward_islands_colony_1671_1816 -> stkitts_nevis_anguilla_1816_1833: EVOLVED_INTO → PARTITIONED_INTO — colony split into two sub-groupings
  OK: bengal_presidency_crown_1858_1912 -> assam_province_1874_1905: EVOLVED_INTO → PARTITIONED_INTO — Assam carved out while Bengal continued
  OK: bengal_presidency_crown_1858_1912 -> eastern_bengal_and_assam_1905_1912: EVOLVED_INTO → PARTITIONED_INTO — Bengal divided 1905; parent continued as Bengal Partitioned
  OK: bengal_presidency_crown_1858_1912 -> bengal_province_pa

In [14]:
# Cell 8: Add ADMINISTERED_UNDER edges
# New edge type for territories sharing a governor but legally distinct from the administering colony.
# Gold Coast administered Ashanti (conquered 1901), British Togoland (League mandate 1919),
# and Northern Territories (protectorate 1897). All became Ghana together in 1957.
# Note: the old PARTITIONED_INTO edges between Gold Coast and Ashanti/Togoland were deleted in Cell 6.

ADMINISTERED_UNDER = [
    ('gold_coast_colony_1874_1957', 'ashanti_1901_1957',
     'Ashanti conquered 1901, administered by Gold Coast governor'),
    ('gold_coast_colony_1874_1957', 'british_togoland_1919_1957',
     'League of Nations mandate 1919, administered by Gold Coast'),
    ('gold_coast_colony_1874_1957', 'northern_territories_gold_coast_1897_1957',
     'Northern Territories protectorate 1897, administered by Gold Coast'),
]

for src, tgt, reason in ADMINISTERED_UNDER:
    result = run_write("""
        MATCH (s:Colony {colony_id: $src}), (t:Colony {colony_id: $tgt})
        MERGE (s)-[:ADMINISTERED_UNDER]->(t)
    """, src=src, tgt=tgt)
    status = "OK" if "rels_created" in result else "CHECK (nodes missing or edge exists)"
    print(f"  {status}: {src} -> {tgt} — {reason}")

print(f"\nProcessed {len(ADMINISTERED_UNDER)} ADMINISTERED_UNDER edges.")
print("Note: Northern Territories node must be created in Cell 9 first if not already present.")

  OK: gold_coast_colony_1874_1957 -> ashanti_1901_1957 — Ashanti conquered 1901, administered by Gold Coast governor
  OK: gold_coast_colony_1874_1957 -> british_togoland_1919_1957 — League of Nations mandate 1919, administered by Gold Coast
  CHECK (nodes missing or edge exists): gold_coast_colony_1874_1957 -> northern_territories_gold_coast_1897_1957 — Northern Territories protectorate 1897, administered by Gold Coast

Processed 3 ADMINISTERED_UNDER edges.
Note: Northern Territories node must be created in Cell 9 first if not already present.


In [17]:
# Cell 9: Add missing nodes
# Nodes that exist in the visualization (HTML patches) but not in the KG.
# Run this BEFORE Cell 10 (missing edges) since edges reference these nodes.

NEW_NODES = [
    # North America
    {
        'colony_id': 'nova_scotia_pre_1713_1784', 'canonical_name': 'Nova Scotia',
        'established_year': 1713, 'independence_year': 1784,
        'region': 'North America', 'type': 'Crown Colony', 'wikidata_id': 'Q98931415',
    },
    {
        'colony_id': 'nova_scotia_post_1784_1867', 'canonical_name': 'Nova Scotia',
        'established_year': 1784, 'independence_year': 1867,
        'region': 'North America', 'type': 'Crown Colony', 'wikidata_id': 'Q98931415',
    },
    {
        'colony_id': 'red_river_colony_1811_1870', 'canonical_name': 'Red River Colony',
        'established_year': 1811, 'independence_year': 1870,
        'region': 'North America', 'type': 'Colony', 'wikidata_id': 'Q1143638',
    },
    {
        'colony_id': 'united_states_1776', 'canonical_name': 'United States of America',
        'established_year': 1776, 'independence_year': None,
        'region': 'North America', 'type': 'Independence', 'wikidata_id': 'Q30',
    },
    {
        'colony_id': 'canada_independent_1931', 'canonical_name': 'Canada',
        'established_year': 1931, 'independence_year': None,
        'region': 'North America', 'type': 'Independence', 'wikidata_id': 'Q16',
    },
    {
        'colony_id': 'newfoundland_commission_1934_1949', 'canonical_name': 'Newfoundland (Commission)',
        'established_year': 1934, 'independence_year': 1949,
        'region': 'North America', 'type': 'Crown Colony', 'wikidata_id': 'Q5152678',
    },

    # Pacific / Australasia
    {
        'colony_id': 'australia_independent_1942', 'canonical_name': 'Australia',
        'established_year': 1942, 'independence_year': None,
        'region': 'Pacific', 'type': 'Independence', 'wikidata_id': 'Q408',
    },
    {
        'colony_id': 'new_zealand_colony_1840_1907', 'canonical_name': 'New Zealand Colony',
        'established_year': 1840, 'independence_year': 1907,
        'region': 'Pacific', 'type': 'Crown Colony', 'wikidata_id': 'Q664',
    },
    {
        'colony_id': 'dominion_of_new_zealand_1907_1947', 'canonical_name': 'Dominion of New Zealand',
        'established_year': 1907, 'independence_year': 1947,
        'region': 'Pacific', 'type': 'Dominion', 'wikidata_id': 'Q664',
    },
    {
        'colony_id': 'new_zealand_independent_1947', 'canonical_name': 'New Zealand',
        'established_year': 1947, 'independence_year': None,
        'region': 'Pacific', 'type': 'Independence', 'wikidata_id': 'Q664',
    },

    # Southern Africa — post-federation nodes
    {
        'colony_id': 'southern_rhodesia_post_federation_1963', 'canonical_name': 'Southern Rhodesia',
        'established_year': 1963, 'independence_year': 1965,
        'region': 'Southern Africa', 'type': 'Self-governing Colony', 'wikidata_id': 'Q750583',
    },
    {
        'colony_id': 'northern_rhodesia_post_federation_1964', 'canonical_name': 'Northern Rhodesia',
        'established_year': 1963, 'independence_year': 1964,
        'region': 'Southern Africa', 'type': 'Transitional', 'wikidata_id': 'Q953903',
    },

    # West Africa — Northern Territories of the Gold Coast
    {
        'colony_id': 'northern_territories_gold_coast_1897_1957', 'canonical_name': 'Northern Territories of the Gold Coast',
        'established_year': 1897, 'independence_year': 1957,
        'region': 'West Africa', 'type': 'Protectorate', 'wikidata_id': 'Q2000610',
    },

    # East Africa — Tanzania
    {
        'colony_id': 'tanzania_1964', 'canonical_name': 'Tanzania',
        'established_year': 1964, 'independence_year': None,
        'region': 'East Africa', 'type': 'Independence', 'wikidata_id': 'Q924',
    },

    # Southeast Asia — Malaysia and Singapore independent
    {
        'colony_id': 'malaysia_1963', 'canonical_name': 'Malaysia',
        'established_year': 1963, 'independence_year': None,
        'region': 'Southeast Asia', 'type': 'Independence', 'wikidata_id': 'Q833',
    },
    {
        'colony_id': 'singapore_independent_1965', 'canonical_name': 'Singapore',
        'established_year': 1965, 'independence_year': None,
        'region': 'Southeast Asia', 'type': 'Independence', 'wikidata_id': 'Q334',
    },

    # South Asia — missing Indian provinces
    {
        'colony_id': 'coorg_province_1834_1947', 'canonical_name': 'Coorg',
        'established_year': 1834, 'independence_year': 1947,
        'region': 'South Asia', 'type': 'Province', 'wikidata_id': 'Q864930',
    },
    {
        'colony_id': 'ajmer_merwara_1818_1947', 'canonical_name': 'Ajmer-Merwara',
        'established_year': 1818, 'independence_year': 1947,
        'region': 'South Asia', 'type': 'Province', 'wikidata_id': 'Q4700553',
    },
    {
        'colony_id': 'sind_province_1843_1947', 'canonical_name': 'Sind',
        'established_year': 1936, 'independence_year': 1947,
        'region': 'South Asia', 'type': 'Province', 'wikidata_id': 'Q7523949',
    },
    {
        'colony_id': 'andaman_and_nicobar_islands_1789_1947', 'canonical_name': 'Andaman and Nicobar Islands',
        'established_year': 1789, 'independence_year': 1947,
        'region': 'South Asia', 'type': 'Province', 'wikidata_id': 'Q40888',
    },

    # Europe — Minorca (3 periods), Corsica, Heligoland, Ireland chain
    {
        'colony_id': 'minorca_first_1708_1756', 'canonical_name': 'Minorca (1st British)',
        'established_year': 1708, 'independence_year': 1756,
        'region': 'Europe', 'type': 'Crown Colony', 'wikidata_id': 'Q173095',
    },
    {
        'colony_id': 'minorca_second_1763_1782', 'canonical_name': 'Minorca (2nd British)',
        'established_year': 1763, 'independence_year': 1782,
        'region': 'Europe', 'type': 'Crown Colony', 'wikidata_id': 'Q173095',
    },
    {
        'colony_id': 'minorca_third_1798_1802', 'canonical_name': 'Minorca (3rd British)',
        'established_year': 1798, 'independence_year': 1802,
        'region': 'Europe', 'type': 'Crown Colony', 'wikidata_id': 'Q173095',
    },
    {
        'colony_id': 'corsica_1794_1796', 'canonical_name': 'Anglo-Corsican Kingdom',
        'established_year': 1794, 'independence_year': 1796,
        'region': 'Europe', 'type': 'Protectorate', 'wikidata_id': 'Q2259576',
    },
    {
        'colony_id': 'heligoland_1807_1890', 'canonical_name': 'Heligoland',
        'established_year': 1807, 'independence_year': 1890,
        'region': 'Europe', 'type': 'Crown Colony', 'wikidata_id': 'Q3104',
    },
    {
        'colony_id': 'kingdom_of_ireland_1541_1801', 'canonical_name': 'Kingdom of Ireland',
        'established_year': 1541, 'independence_year': 1801,
        'region': 'Europe', 'type': 'Kingdom', 'wikidata_id': 'Q179876',
    },
    {
        'colony_id': 'ireland_uk_1801_1922', 'canonical_name': 'Ireland (UK)',
        'established_year': 1801, 'independence_year': 1922,
        'region': 'Europe', 'type': 'Union', 'wikidata_id': 'Q174193',
    },
    {
        'colony_id': 'irish_free_state_1922_1937', 'canonical_name': 'Irish Free State',
        'established_year': 1922, 'independence_year': 1937,
        'region': 'Europe', 'type': 'Dominion', 'wikidata_id': 'Q31747',
    },
    {
        'colony_id': 'northern_ireland_1920', 'canonical_name': 'Northern Ireland',
        'established_year': 1920, 'independence_year': None,
        'region': 'Europe', 'type': 'Devolved Government', 'wikidata_id': 'Q26',
    },
]

for node in NEW_NODES:
    props = {k: v for k, v in node.items() if v is not None}
    set_clauses = ", ".join(f"n.{k} = ${k}" for k in props if k != 'colony_id')
    result = run_write(f"""
        MERGE (n:Colony {{colony_id: $colony_id}})
        ON CREATE SET {set_clauses}
    """, **props)
    status = "CREATED" if "nodes_created" in result else "EXISTS"
    print(f"  {status}: {node['colony_id']} ({node['canonical_name']})")

# Handle Nova Scotia split: delete old node, redirect its edges
# The old nova_scotia_1621_1867 node needs to be replaced by pre/post nodes
old_ns = run("MATCH (n:Colony {colony_id: 'nova_scotia_1621_1867'}) RETURN n.colony_id as id")
if old_ns:
    # Redirect inbound edges to pre-partition node
    run_write("""
        MATCH (src:Colony)-[r]->(old:Colony {colony_id: 'nova_scotia_1621_1867'})
        MATCH (new:Colony {colony_id: 'nova_scotia_pre_1713_1784'})
        WITH src, old, new, type(r) as relType, r
        CALL {
            WITH src, new, relType
            WITH src, new, relType
            FOREACH(_ IN CASE WHEN relType = 'EVOLVED_INTO' THEN [1] ELSE [] END |
                MERGE (src)-[:EVOLVED_INTO]->(new))
            FOREACH(_ IN CASE WHEN relType = 'PARTITIONED_INTO' THEN [1] ELSE [] END |
                MERGE (src)-[:PARTITIONED_INTO]->(new))
        }
        DELETE r
    """)
    # Redirect outbound edges to post-partition node
    run_write("""
        MATCH (old:Colony {colony_id: 'nova_scotia_1621_1867'})-[r]->(tgt:Colony)
        MATCH (new:Colony {colony_id: 'nova_scotia_post_1784_1867'})
        WITH old, tgt, new, type(r) as relType, r
        CALL {
            WITH new, tgt, relType
            WITH new, tgt, relType
            FOREACH(_ IN CASE WHEN relType = 'EVOLVED_INTO' THEN [1] ELSE [] END |
                MERGE (new)-[:EVOLVED_INTO]->(tgt))
            FOREACH(_ IN CASE WHEN relType = 'PARTITIONED_INTO' THEN [1] ELSE [] END |
                MERGE (new)-[:PARTITIONED_INTO]->(tgt))
        }
        DELETE r
    """)
    # Delete old node
    run_write("MATCH (n:Colony {colony_id: 'nova_scotia_1621_1867'}) DETACH DELETE n")
    print("  SPLIT: nova_scotia_1621_1867 → pre (1713-1784) + post (1784-1867)")
else:
    print("  SKIP: nova_scotia_1621_1867 not found (already split or deleted)")

# Handle New Zealand split: delete old node, redirect edges
old_nz = run("MATCH (n:Colony {colony_id: 'new_zealand_1840_1947'}) RETURN n.colony_id as id")
if old_nz:
    # Redirect inbound edges to colony node
    run_write("""
        MATCH (src:Colony)-[r]->(old:Colony {colony_id: 'new_zealand_1840_1947'})
        MATCH (new:Colony {colony_id: 'new_zealand_colony_1840_1907'})
        WITH src, new, type(r) as relType, r
        CALL {
            WITH src, new, relType
            WITH src, new, relType
            FOREACH(_ IN CASE WHEN relType = 'EVOLVED_INTO' THEN [1] ELSE [] END |
                MERGE (src)-[:EVOLVED_INTO]->(new))
            FOREACH(_ IN CASE WHEN relType = 'PARTITIONED_INTO' THEN [1] ELSE [] END |
                MERGE (src)-[:PARTITIONED_INTO]->(new))
        }
        DELETE r
    """)
    # Redirect outbound edges to dominion node (the middle of the chain)
    run_write("""
        MATCH (old:Colony {colony_id: 'new_zealand_1840_1947'})-[r]->(tgt:Colony)
        MATCH (new:Colony {colony_id: 'dominion_of_new_zealand_1907_1947'})
        WITH old, tgt, new, type(r) as relType, r
        CALL {
            WITH new, tgt, relType
            WITH new, tgt, relType
            FOREACH(_ IN CASE WHEN relType = 'EVOLVED_INTO' THEN [1] ELSE [] END |
                MERGE (new)-[:EVOLVED_INTO]->(tgt))
            FOREACH(_ IN CASE WHEN relType = 'PARTITIONED_INTO' THEN [1] ELSE [] END |
                MERGE (new)-[:PARTITIONED_INTO]->(tgt))
        }
        DELETE r
    """)
    # Delete old node
    run_write("MATCH (n:Colony {colony_id: 'new_zealand_1840_1947'}) DETACH DELETE n")
    print("  SPLIT: new_zealand_1840_1947 → Colony (1840-1907) + Dominion (1907-1947) + Independent (1947-)")
else:
    print("  SKIP: new_zealand_1840_1947 not found (already split or deleted)")

print(f"\nProcessed {len(NEW_NODES)} new nodes + 2 node splits.")

  CREATED: nova_scotia_pre_1713_1784 (Nova Scotia)
  CREATED: nova_scotia_post_1784_1867 (Nova Scotia)
  CREATED: red_river_colony_1811_1870 (Red River Colony)
  CREATED: united_states_1776 (United States of America)
  CREATED: canada_independent_1931 (Canada)
  CREATED: newfoundland_commission_1934_1949 (Newfoundland (Commission))
  CREATED: australia_independent_1942 (Australia)
  CREATED: new_zealand_colony_1840_1907 (New Zealand Colony)
  CREATED: dominion_of_new_zealand_1907_1947 (Dominion of New Zealand)
  CREATED: new_zealand_independent_1947 (New Zealand)
  CREATED: southern_rhodesia_post_federation_1963 (Southern Rhodesia)
  CREATED: northern_rhodesia_post_federation_1964 (Northern Rhodesia)
  CREATED: northern_territories_gold_coast_1897_1957 (Northern Territories of the Gold Coast)
  CREATED: tanzania_1964 (Tanzania)
  CREATED: malaysia_1963 (Malaysia)
  CREATED: singapore_independent_1965 (Singapore)
  CREATED: coorg_province_1834_1947 (Coorg)
  CREATED: ajmer_merwara_1818_

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: CALL subquery without a variable scope clause is now deprecated. Use CALL (src, new, relType) { ... }} {position: line: 5, column: 9, offset: 209} for query: "\n        MATCH (src:Colony)-[r]->(old:Colony {colony_id: 'nova_scotia_1621_1867'})\n        MATCH (new:Colony {colony_id: 'nova_scotia_pre_1713_1784'})\n        WITH src, old, new, type(r) as relType, r\n        CALL {\n            WITH src, new, relType\n            WITH src, new, relType\n            FOREACH(_ IN CASE WHEN relType = 'EVOLVED_INTO' THEN [1] ELSE [] END |\n                MERGE (src)-[:EVOLVED_INTO]->(new))\n            FOREACH(_ IN CASE WHEN relType = 'PARTITIONED_INTO' THEN [1] ELSE [] END |\n                MERGE (src)-[:PARTITIONED_INTO]->(new))\n        }\n        DELETE

  SPLIT: nova_scotia_1621_1867 → pre (1713-1784) + post (1784-1867)


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: CALL subquery without a variable scope clause is now deprecated. Use CALL (src, new, relType) { ... }} {position: line: 5, column: 9, offset: 207} for query: "\n        MATCH (src:Colony)-[r]->(old:Colony {colony_id: 'new_zealand_1840_1947'})\n        MATCH (new:Colony {colony_id: 'new_zealand_colony_1840_1907'})\n        WITH src, new, type(r) as relType, r\n        CALL {\n            WITH src, new, relType\n            WITH src, new, relType\n            FOREACH(_ IN CASE WHEN relType = 'EVOLVED_INTO' THEN [1] ELSE [] END |\n                MERGE (src)-[:EVOLVED_INTO]->(new))\n            FOREACH(_ IN CASE WHEN relType = 'PARTITIONED_INTO' THEN [1] ELSE [] END |\n                MERGE (src)-[:PARTITIONED_INTO]->(new))\n        }\n        DELETE r

  SPLIT: new_zealand_1840_1947 → Colony (1840-1907) + Dominion (1907-1947) + Independent (1947-)

Processed 29 new nodes + 2 node splits.


In [18]:
# Cell 10: Add missing edges
# All edges from the HTML patches that don't exist in the source KG.
# Format: (source_id, target_id, type, reason)

NEW_EDGES = [
    # === North America ===
    # 13 colonies → USA
    ('virginia_colony_1607_1783', 'united_states_1776', 'EVOLVED_INTO', '13 colonies → independence'),
    ('massachusetts_bay_colony_1630_1783', 'united_states_1776', 'EVOLVED_INTO', '13 colonies → independence'),
    ('pennsylvania_colony_1681_1783', 'united_states_1776', 'EVOLVED_INTO', '13 colonies → independence'),
    ('new_york_colony_1664_1783', 'united_states_1776', 'EVOLVED_INTO', '13 colonies → independence'),
    ('maryland_colony_1634_1783', 'united_states_1776', 'EVOLVED_INTO', '13 colonies → independence'),
    ('connecticut_colony_1636_1783', 'united_states_1776', 'EVOLVED_INTO', '13 colonies → independence'),
    ('rhode_island_colony_1636_1783', 'united_states_1776', 'EVOLVED_INTO', '13 colonies → independence'),
    ('delaware_colony_1664_1783', 'united_states_1776', 'EVOLVED_INTO', '13 colonies → independence'),
    ('new_hampshire_colony_1623_1783', 'united_states_1776', 'EVOLVED_INTO', '13 colonies → independence'),
    ('new_jersey_colony_1664_1783', 'united_states_1776', 'EVOLVED_INTO', '13 colonies → independence'),
    ('north_carolina_colony_1663_1783', 'united_states_1776', 'EVOLVED_INTO', '13 colonies → independence'),
    ('south_carolina_colony_1663_1783', 'united_states_1776', 'EVOLVED_INTO', '13 colonies → independence'),
    ('georgia_colony_1732_1783', 'united_states_1776', 'EVOLVED_INTO', '13 colonies → independence'),

    # Canada chain
    ('canada_dominion_of_1867_ongoing', 'canada_independent_1931', 'EVOLVED_INTO', 'Statute of Westminster 1931'),

    # New France → Quebec
    ('new_france_1608_1763', 'province_of_quebec_british_1763_1791', 'EVOLVED_INTO', 'Treaty of Paris 1763'),

    # Île Royale → Nova Scotia (Treaty of Paris 1763)
    ('ile_royale_1713_1763', 'nova_scotia_pre_1713_1784', 'EVOLVED_INTO', 'Treaty of Paris 1763'),

    # Acadia → Nova Scotia (Treaty of Utrecht 1713)
    ('acadia_french_1604_1713', 'nova_scotia_pre_1713_1784', 'EVOLVED_INTO', 'Treaty of Utrecht 1713'),

    # Nova Scotia partition (1784)
    ('nova_scotia_pre_1713_1784', 'nova_scotia_post_1784_1867', 'PARTITIONED_INTO', 'NS continued after NB carved out'),
    ('nova_scotia_pre_1713_1784', 'new_brunswick_1784_1867', 'PARTITIONED_INTO', 'NB carved from NS 1784'),
    ('nova_scotia_post_1784_1867', 'canada_dominion_of_1867_ongoing', 'EVOLVED_INTO', 'Confederation 1867'),

    # Rupert's Land splits
    ('ruperts_land_1670_1870', 'north_western_territory_1670_1870', 'PARTITIONED_INTO', 'NWT was HBC territory'),
    ('ruperts_land_1670_1870', 'red_river_colony_1811_1870', 'PARTITIONED_INTO', 'Selkirk settlement carved out'),
    ('ruperts_land_1670_1870', 'canada_dominion_of_1867_ongoing', 'EVOLVED_INTO', 'HBC surrendered to Canada 1870'),
    ('red_river_colony_1811_1870', 'canada_dominion_of_1867_ongoing', 'EVOLVED_INTO', 'became Manitoba 1870'),
    ('north_western_territory_1670_1870', 'canada_dominion_of_1867_ongoing', 'EVOLVED_INTO', 'transferred to Canada 1870'),

    # Newfoundland chain
    ('colony_of_newfoundland_1610_1949', 'dominion_of_newfoundland_1907_1934', 'EVOLVED_INTO', 'Colony → Dominion 1907'),
    ('dominion_of_newfoundland_1907_1934', 'newfoundland_commission_1934_1949', 'EVOLVED_INTO', 'Dominion → Commission 1934'),
    ('newfoundland_commission_1934_1949', 'canada_independent_1931', 'EVOLVED_INTO', 'joined Canada 1949'),

    # BC: Vancouver Island + Mainland merged into United Colony
    ('vancouver_island_1849_1866', 'united_colony_of_bc_1866_1871', 'EVOLVED_INTO', 'merged 1866'),
    ('mainland_british_columbia_1858_1866', 'united_colony_of_bc_1866_1871', 'EVOLVED_INTO', 'merged 1866'),

    # === Pacific / Australasia ===
    # Australia: Dominion → Independent
    ('commonwealth_of_australia_1901_ongoing', 'australia_independent_1942', 'EVOLVED_INTO', 'Statute of Westminster Adoption Act 1942'),

    # New Zealand chain
    ('new_zealand_colony_1840_1907', 'dominion_of_new_zealand_1907_1947', 'EVOLVED_INTO', 'Colony → Dominion 1907'),
    ('dominion_of_new_zealand_1907_1947', 'new_zealand_independent_1947', 'EVOLVED_INTO', 'Statute of Westminster Adoption Act 1947'),

    # === Southern Africa ===
    # Post-federation Rhodesia
    ('federation_of_rhodesia_and_nyasaland_1953_1963', 'southern_rhodesia_post_federation_1963', 'PARTITIONED_INTO', 'Federation dissolved 1963'),
    ('federation_of_rhodesia_and_nyasaland_1953_1963', 'northern_rhodesia_post_federation_1964', 'PARTITIONED_INTO', 'Federation dissolved 1963'),
    ('federation_of_rhodesia_and_nyasaland_1953_1963', 'nyasaland_post_federation_1963', 'PARTITIONED_INTO', 'Federation dissolved 1963'),
    ('southern_rhodesia_post_federation_1963', 'rhodesia_udi_1965_1979', 'EVOLVED_INTO', 'SR → UDI 1965'),

    # Northern Rhodesia → Zambia (independence 1964)
    # (edge from NR post-federation to Zambia if Zambia node exists)

    # === East Africa ===
    # Tanzania merger
    ('tanganyika_1961_1964', 'tanzania_1964', 'EVOLVED_INTO', 'merged with Zanzibar 1964'),
    ('zanzibar_independent_1963_1964', 'tanzania_1964', 'EVOLVED_INTO', 'merged with Tanganyika 1964'),

    # === South Asia ===
    # Fort succession
    ('factory_at_surat_1612_1857', 'bombay_presidency_company_1687_1858', 'EVOLVED_INTO', 'EIC moved HQ to Bombay 1687'),
    ('fort_william_calcutta_1696_1857', 'bengal_presidency_company_1757_1858', 'EVOLVED_INTO', 'Battle of Plassey 1757'),

    # Assam under Bengal
    ('bengal_presidency_company_1757_1858', 'assam_bengal_presidency_1826_1874', 'PARTITIONED_INTO', 'Assam administered under Bengal'),

    # Baluchistan → Pakistan
    ('baluchistan_1877_1947', 'dominion_of_pakistan_1947_1956', 'EVOLVED_INTO', 'independence 1947'),

    # Aden Province → Aden Colony
    ('aden_1839_1963', 'aden_colony_1937_1967', 'EVOLVED_INTO', 'separated as Crown Colony 1937'),

    # Indian provinces → India
    ('coorg_province_1834_1947', 'dominion_of_india_1947_1950', 'EVOLVED_INTO', 'independence 1947'),
    ('ajmer_merwara_1818_1947', 'dominion_of_india_1947_1950', 'EVOLVED_INTO', 'independence 1947'),
    ('andaman_and_nicobar_islands_1789_1947', 'dominion_of_india_1947_1950', 'EVOLVED_INTO', 'independence 1947'),

    # Sind carved from Bombay → Pakistan
    ('bombay_presidency_crown_1858_1947', 'sind_province_1843_1947', 'PARTITIONED_INTO', 'Sind separated 1936'),
    ('sind_province_1843_1947', 'dominion_of_pakistan_1947_1956', 'EVOLVED_INTO', 'independence 1947'),

    # === Southeast Asia ===
    # Malaysia formation
    ('malaya_independent_1957_1963', 'malaysia_1963', 'EVOLVED_INTO', 'Malaysia formed 1963'),
    ('singapore_crown_colony_1946_1963', 'malaysia_1963', 'EVOLVED_INTO', 'joined Malaysia 1963'),
    ('north_borneo_crown_colony_1946_1963', 'malaysia_1963', 'EVOLVED_INTO', 'joined Malaysia 1963'),
    ('sarawak_crown_colony_1946_1963', 'malaysia_1963', 'EVOLVED_INTO', 'joined Malaysia 1963'),
    ('labuan_1846_1963', 'malaysia_1963', 'EVOLVED_INTO', 'joined Malaysia 1963'),
    ('malaysia_1963', 'singapore_independent_1965', 'PARTITIONED_INTO', 'Singapore separated 1965'),

    # === Europe ===
    # Minorca succession
    ('minorca_first_1708_1756', 'minorca_second_1763_1782', 'EVOLVED_INTO', 'same island recaptured'),
    ('minorca_second_1763_1782', 'minorca_third_1798_1802', 'EVOLVED_INTO', 'same island recaptured'),

    # Ireland chain
    ('kingdom_of_ireland_1541_1801', 'ireland_uk_1801_1922', 'EVOLVED_INTO', 'Acts of Union 1800'),
    ('ireland_uk_1801_1922', 'irish_free_state_1922_1937', 'PARTITIONED_INTO', 'partition of Ireland 1920-22'),
    ('ireland_uk_1801_1922', 'northern_ireland_1920', 'PARTITIONED_INTO', 'partition of Ireland 1920-22'),
]

for src, tgt, etype, reason in NEW_EDGES:
    result = run_write(f"""
        MATCH (s:Colony {{colony_id: $src}}), (t:Colony {{colony_id: $tgt}})
        MERGE (s)-[:{etype}]->(t)
    """, src=src, tgt=tgt)
    if "rels_created" in result:
        print(f"  CREATED: {src} -[{etype}]-> {tgt}")
    elif "no changes" in result:
        # Check if both nodes exist
        check = run("MATCH (s:Colony {colony_id: $src}), (t:Colony {colony_id: $tgt}) RETURN s.colony_id, t.id", src=src, tgt=tgt)
        if check:
            print(f"  EXISTS:  {src} -[{etype}]-> {tgt}")
        else:
            s_exists = run("MATCH (n:Colony {colony_id: $id}) RETURN n.colony_id", id=src)
            t_exists = run("MATCH (n:Colony {colony_id: $id}) RETURN n.colony_id", id=tgt)
            print(f"  MISSING: {src} ({'found' if s_exists else 'NOT FOUND'}) -> {tgt} ({'found' if t_exists else 'NOT FOUND'})")

print(f"\nProcessed {len(NEW_EDGES)} new edges.")

  CREATED: virginia_colony_1607_1783 -[EVOLVED_INTO]-> united_states_1776
  CREATED: massachusetts_bay_colony_1630_1783 -[EVOLVED_INTO]-> united_states_1776
  CREATED: pennsylvania_colony_1681_1783 -[EVOLVED_INTO]-> united_states_1776
  CREATED: new_york_colony_1664_1783 -[EVOLVED_INTO]-> united_states_1776
  CREATED: maryland_colony_1634_1783 -[EVOLVED_INTO]-> united_states_1776
  CREATED: connecticut_colony_1636_1783 -[EVOLVED_INTO]-> united_states_1776
  CREATED: rhode_island_colony_1636_1783 -[EVOLVED_INTO]-> united_states_1776
  CREATED: delaware_colony_1664_1783 -[EVOLVED_INTO]-> united_states_1776
  CREATED: new_hampshire_colony_1623_1783 -[EVOLVED_INTO]-> united_states_1776
  CREATED: new_jersey_colony_1664_1783 -[EVOLVED_INTO]-> united_states_1776
  CREATED: north_carolina_colony_1663_1783 -[EVOLVED_INTO]-> united_states_1776
  CREATED: south_carolina_colony_1663_1783 -[EVOLVED_INTO]-> united_states_1776
  CREATED: georgia_colony_1732_1783 -[EVOLVED_INTO]-> united_states_1776


In [20]:
# Cell 11: Region assignments
# Update region property for nodes that need sub-region mapping.
# This applies the AFRICA_SUBREGIONS mapping and other region fixes from the HTML.

REGION_FIXES = {
    # America → North America
    'settlement_of_belize_1798_1862': 'Caribbean',
    'british_honduras_1862_1981': 'Caribbean',
    'antigua_montserrat_barbuda_1816_1833': 'Caribbean',
    'stkitts_nevis_anguilla_1816_1833': 'Caribbean',
    'bermuda_1609_ongoing': 'Atlantic',
    'british_guiana_1831_1966': 'South America',

    # Southern Africa
    'cape_colony_dutch_1652_1795': 'Southern Africa',
    'cape_colony_british_1795_1803_1795_1803': 'Southern Africa',
    'cape_colony_dutch_restored_1803_1806': 'Southern Africa',
    'cape_colony_british_1806_1910': 'Southern Africa',
    'natalia_republic_1839_1843': 'Southern Africa',
    'colony_of_natal_1843_1910': 'Southern Africa',
    'orange_river_sovereignty_1848_1854': 'Southern Africa',
    'south_african_republic_transvaal_1852_1877': 'Southern Africa',
    'orange_free_state_1854_1900': 'Southern Africa',
    'basutoland_1868_1966': 'Southern Africa',
    'griqualand_west_1871_1880': 'Southern Africa',
    'transvaal_colony_first_british_1877_1881': 'Southern Africa',
    'south_african_republic_restored_1881_1900': 'Southern Africa',
    'bechuanaland_protectorate_1884_1966': 'Southern Africa',
    'british_bechuanaland_1885_1895': 'Southern Africa',
    'zululand_1887_1897': 'Southern Africa',
    'british_south_africa_company_territory_1889_1923': 'Southern Africa',
    'orange_river_colony_1900_1910': 'Southern Africa',
    'transvaal_colony_second_british_1900_1910': 'Southern Africa',
    'swaziland_1903_1968': 'Southern Africa',
    'union_of_south_africa_1910_1961': 'Southern Africa',
    'southern_rhodesia_colony_1923_1953': 'Southern Africa',
    'northern_rhodesia_colony_1924_1953': 'Southern Africa',
    'federation_of_rhodesia_and_nyasaland_1953_1963': 'Southern Africa',
    'rhodesia_udi_1965_1979': 'Southern Africa',
    'zimbabwe_rhodesia_1979_1979': 'Southern Africa',
    'southern_rhodesia_restored_1979_1980': 'Southern Africa',
    'british_central_africa_protectorate_1891_1907': 'Southern Africa',
    'nyasaland_1891_1964': 'Southern Africa',
    'nyasaland_post_federation_1963': 'Southern Africa',

    # West Africa
    'james_island_trading_post_1661_1816': 'West Africa',
    'cape_coast_castle_1664_1821': 'West Africa',
    'gambia_colony_1816_1888': 'West Africa',
    'british_gold_coast_1821_1874': 'West Africa',
    'gold_coast_colony_1874_1957': 'West Africa',
    'oil_rivers_protectorate_1885_1893': 'West Africa',
    'royal_niger_company_territory_1886_1900': 'West Africa',
    'lagos_protectorate_1887_1906': 'West Africa',
    'gambia_colony_and_protectorate_1816_1965': 'West Africa',
    'niger_coast_protectorate_1893_1900': 'West Africa',
    'ashanti_1901_1957': 'West Africa',
    'southern_nigeria_protectorate_1900_1914': 'West Africa',
    'northern_nigeria_1900_1914': 'West Africa',
    'colony_and_protectorate_of_nigeria_1914_1960': 'West Africa',
    'british_cameroons_1916_1961': 'West Africa',
    'british_togoland_1919_1957': 'West Africa',
    'province_of_freedom_1787_1792': 'West Africa',
    'sierra_leone_company_settlement_1792_1808': 'West Africa',
    'sierra_leone_colony_1808_1896': 'West Africa',
    'sierra_leone_colony_and_protectorate_1787_1961': 'West Africa',

    # East Africa
    'imperial_british_east_africa_company_territory_1888_1895': 'East Africa',
    'zanzibar_1890_1963': 'East Africa',
    'uganda_1894_1962': 'East Africa',
    'east_africa_protectorate_1895_1920': 'East Africa',
    'german_east_africa_british_occupied_1916_1922': 'East Africa',
    'kenya_colony_and_protectorate_of_1920_1963': 'East Africa',
    'tanganyika_territory_1922_1961': 'East Africa',
    'tanganyika_1961_1964': 'East Africa',
    'zanzibar_independent_1963_1964': 'East Africa',

    # Africa (Islands)
    'fernando_po_british_1827_1843': 'Africa (Islands)',
    'st._helena_1659_ongoing': 'Africa (Islands)',
    'ascension_island_1815_ongoing': 'Africa (Islands)',
    'tristan_da_cunha_1816_ongoing': 'Africa (Islands)',
    'isle_de_france_british_occupation_1810_1814': 'Africa (Islands)',
    'mauritius_1814_1968': 'Africa (Islands)',
    'seychelles_mauritius_dependency_1814_1903': 'Africa (Islands)',
    'rodrigues_1814_1968': 'Africa (Islands)',
    'seychelles_1903_1976': 'Africa (Islands)',
    'aldabra_islands_1903_1965': 'Africa (Islands)',
    'farquhar_islands_1903_1965': 'Africa (Islands)',
    'desroches_islands_1903_1965': 'Africa (Islands)',

    # Middle East
    'egypt_1882_1922': 'Middle East',
    'sudan_anglo-egyptian_1899_1956': 'Middle East',
    'british_somaliland_1884_1960': 'Middle East',
    'aden_1839_1963': 'Middle East',
    'aden_colony_1937_1967': 'Middle East',
    'aden_protectorate_1839_1967': 'Middle East',
    'south_yemen_1967_1990': 'Middle East',
    'socotra_island_1886_1967': 'Middle East',
    'bahrain_protectorate_1861_1971': 'Middle East',
    'kuwait_1899_1961': 'Middle East',
    'qatar_protectorate_1916_1971': 'Middle East',
    'trucial_states_1820_1971': 'Middle East',
    'muscat_and_oman_protectorate_1891_1971': 'Middle East',
    'palestine_1920_1948': 'Middle East',
    'transjordan_1922_1946': 'Middle East',
    'mesopotamia_british_occupied_1914_1920': 'Middle East',
    'mandatory_iraq_1920_1932': 'Middle East',
    'iraq_1932_ongoing': 'Middle East',

    # South Asia
    'bengal_presidency_company_1757_1858': 'South Asia',
    'madras_presidency_company_1640_1858': 'South Asia',
    'bombay_presidency_company_1687_1858': 'South Asia',
    'bengal_presidency_crown_1858_1912': 'South Asia',
    'madras_presidency_crown_1858_1947': 'South Asia',
    'bombay_presidency_crown_1858_1947': 'South Asia',
    'punjab_province_1849_1947': 'South Asia',
    'central_provinces_1861_1947': 'South Asia',
    'united_provinces_1877_1947': 'South Asia',
    'assam_province_1874_1905': 'South Asia',
    'north_west_frontier_province_1901_1947': 'South Asia',
    'eastern_bengal_and_assam_1905_1912': 'South Asia',
    'bengal_province_partitioned_1905_1912': 'South Asia',
    'bengal_province_reunited_1912_1947': 'South Asia',
    'bihar_and_orissa_1912_1947': 'South Asia',
    'assam_bengal_presidency_1826_1874': 'South Asia',
    'baluchistan_1877_1947': 'South Asia',
    'assam_province_restored_1912_1947': 'South Asia',
    'dominion_of_india_1947_1950': 'South Asia',
    'dominion_of_pakistan_1947_1956': 'South Asia',
    'lower_burma_1824_1886': 'South Asia',
    'burma_british_india_1886_1937': 'South Asia',
    'burma_separate_colony_1937_1948': 'South Asia',
    'ceylon_1795_1948': 'South Asia',
    'factory_at_surat_1612_1857': 'South Asia',
    'fort_william_calcutta_1696_1857': 'South Asia',
    'maldives_protectorate_1887_1965': 'South Asia',
    'chagos_islands_mauritius_dependency_1814_1965': 'South Asia',

    # Southeast Asia
    'hong_kong_1841_1997': 'Southeast Asia',
    'penang_prince_of_wales_island_1786_1826': 'Southeast Asia',
    'singapore_settlement_1819_1826': 'Southeast Asia',
    'malacca_settlement_1824_1826': 'Southeast Asia',
    'straits_settlements_1826_1946': 'Southeast Asia',
    'labuan_1846_1963': 'Southeast Asia',
    'federated_malay_states_1895_1946': 'Southeast Asia',
    'unfederated_malay_states_1909_1946': 'Southeast Asia',
    'british_north_borneo_1882_1946': 'Southeast Asia',
    'north_borneo_crown_colony_1946_1963': 'Southeast Asia',
    'sarawak_1841_1946': 'Southeast Asia',
    'sarawak_crown_colony_1946_1963': 'Southeast Asia',
    'malayan_union_1946_1948': 'Southeast Asia',
    'federation_of_malaya_1948_1957': 'Southeast Asia',
    'malaya_independent_1957_1963': 'Southeast Asia',
    'singapore_crown_colony_1946_1963': 'Southeast Asia',
    'brunei_protectorate_1888_1984': 'Southeast Asia',
    'weihaiwei_1898_1930': 'Southeast Asia',
    'bencoolen_bengkulu_1685_1824': 'Southeast Asia',
    'java_british_occupation_1811_1816': 'Southeast Asia',
    'moluccas_british_occupation_1810_1817': 'Southeast Asia',
    'banda_islands_british_occupation_1810_1817': 'Southeast Asia',
    'netherlands_east_indies_british_administration_1945_1946': 'Southeast Asia',
    'shanghai_international_settlement_1863_1943': 'Southeast Asia',

    # Pacific guano/whaling
    'campbell_island_whaling_1810_1860': 'Pacific (Guano/Whaling)',
    'norfolk_island_whaling_station_1830_1870': 'Pacific (Guano/Whaling)',
    'swains_island_1856_1925': 'Pacific (Guano/Whaling)',
    'howland_island_1857_1898': 'Pacific (Guano/Whaling)',
    'jarvis_island_1857_1898': 'Pacific (Guano/Whaling)',
    'christmas_island_line_islands_1857_1979': 'Pacific (Guano/Whaling)',
    'enderbury_island_1860_1979': 'Pacific (Guano/Whaling)',
    'mckean_island_1860_1979': 'Pacific (Guano/Whaling)',
    'birnie_island_1860_1979': 'Pacific (Guano/Whaling)',
    'rawaki_phoenix_island_1860_1979': 'Pacific (Guano/Whaling)',
    'manra_sydney_island_1860_1979': 'Pacific (Guano/Whaling)',
    'starbuck_island_1866_1979': 'Pacific (Guano/Whaling)',
    'vostok_island_1866_1979': 'Pacific (Guano/Whaling)',
    'flint_island_1866_1979': 'Pacific (Guano/Whaling)',
    'caroline_island_1868_1979': 'Pacific (Guano/Whaling)',
    'baker_island_1855_1898': 'Pacific (Guano/Whaling)',
    'ducie_island_1902_ongoing': 'Pacific (Guano/Whaling)',
    'oeno_island_1902_ongoing': 'Pacific (Guano/Whaling)',
}

# Also fix any nodes with region='America' → 'North America'
run_write("MATCH (n:Colony) WHERE n.region = 'America' SET n.region = 'North America'")

updated = 0
skipped = 0
for nid, region in REGION_FIXES.items():
    result = run_write("MATCH (n:Colony {colony_id: $id}) SET n.region = $region", id=nid, region=region)
    if "props_set" in result:
        updated += 1
    else:
        skipped += 1

print(f"Region assignments: {updated} updated, {skipped} skipped (node not found)")
print("Also fixed any 'America' → 'North America' mappings.")

Region assignments: 165 updated, 0 skipped (node not found)
Also fixed any 'America' → 'North America' mappings.


In [21]:
# Cell 12: Post-fix verification
print("=== POST-FIX STATE ===")

# Node counts by label
for label in ['Colony', 'Territory', 'Place']:
    r = run(f"MATCH (n:{label}) RETURN count(n) as cnt")
    print(f"  {label}: {r[0]['cnt']}")

# Edge counts by type
r = run("""
    MATCH ()-[r]->() 
    WHERE type(r) IN ['EVOLVED_INTO','PARTITIONED_INTO','PART_OF','ADMINISTERED_UNDER'] 
    RETURN type(r) as t, count(r) as cnt ORDER BY t
""")
print("\nEdge counts:")
for row in r:
    print(f"  {row['t']}: {row['cnt']}")

# Region distribution
r = run("MATCH (n:Colony) RETURN n.region as region, count(n) as cnt ORDER BY cnt DESC")
print("\nNodes by region:")
for row in r:
    print(f"  {row['region']}: {row['cnt']}")

# Check for orphan nodes (no edges at all)
r = run("""
    MATCH (n:Colony)
    WHERE NOT (n)--(:Colony) 
    AND n.region IS NOT NULL
    RETURN n.colony_id, n.canonical_name, n.region
    ORDER BY n.region, n.canonical_name
""")
print(f"\nOrphan Colony nodes (no edges): {len(r)}")
for row in r[:20]:
    print(f"  {row['n.region']}: {row['n.canonical_name']} ({row['n.colony_id']})")
if len(r) > 20:
    print(f"  ... and {len(r) - 20} more")

# Check for duplicate IDs (should be 0)
r = run("MATCH (n:Colony) WITH n.colony_id as id, count(*) as cnt WHERE cnt > 1 RETURN id, cnt")
print(f"\nDuplicate IDs: {len(r)}")
for row in r:
    print(f"  {row['id']}: {row['cnt']} copies")

# Spot-check key chains
print("\n=== Spot Checks ===")

# Newfoundland chain
r = run("""
    MATCH path = (a:Colony {colony_id: 'colony_of_newfoundland_1610_1949'})-[:EVOLVED_INTO*]->(z)
    RETURN [n IN nodes(path) | n.canonical_name + ' (' + toString(n.established_year) + ')'] as chain
""")
print("Newfoundland chain:", r[0]['chain'] if r else "NOT FOUND")

# Nova Scotia partition
r = run("""
    MATCH (ns:Colony {colony_id: 'nova_scotia_pre_1713_1784'})-[r]->(child:Colony)
    RETURN child.canonical_name, type(r) as rel
""")
print("Nova Scotia partition:", [(row['child.canonical_name'], row['rel']) for row in r] if r else "NOT FOUND")

# Ireland chain
r = run("""
    MATCH path = (a:Colony {colony_id: 'kingdom_of_ireland_1541_1801'})-[:EVOLVED_INTO|PARTITIONED_INTO*]->(z)
    RETURN [n IN nodes(path) | n.canonical_name] as chain
""")
print("Ireland chains:", [row['chain'] for row in r] if r else "NOT FOUND")

# Gold Coast ADMINISTERED_UNDER
r = run("""
    MATCH (gc:Colony {colony_id: 'gold_coast_colony_1874_1957'})-[:ADMINISTERED_UNDER]->(dep:Colony)
    RETURN dep.canonical_name
""")
print("Gold Coast administers:", [row['dep.canonical_name'] for row in r] if r else "NOT FOUND")

=== POST-FIX STATE ===
  Colony: 303
  Territory: 1
  Place: 6236431

Edge counts:
  ADMINISTERED_UNDER: 2
  EVOLVED_INTO: 174
  PARTITIONED_INTO: 35
  PART_OF: 200867

Nodes by region:
  North America: 48
  Pacific: 37
  Southern Africa: 32
  South Asia: 32
  Southeast Asia: 26
  Caribbean: 24
  West Africa: 21
  Middle East: 18
  Pacific (Guano/Whaling): 18
  Europe: 13
  Africa (Islands): 12
  East Africa: 10
  Antarctica: 7
  Atlantic: 3
  Asia: 1
  South America: 1

Orphan Colony nodes (no edges): 89
  Africa (Islands): Aldabra Islands (aldabra_islands_1903_1965)
  Africa (Islands): Desroches Islands (desroches_islands_1903_1965)
  Africa (Islands): Farquhar Islands (farquhar_islands_1903_1965)
  Africa (Islands): Fernando Po (British) (fernando_po_british_1827_1843)
  Africa (Islands): Rodrigues (rodrigues_1814_1968)
  Africa (Islands): Tristan da Cunha (tristan_da_cunha_1816_ongoing)
  Antarctica: Bouvet Island (Brief Claim) (bouvet_island_brief_claim_1825_1928)
  Antarctica: De

In [ ]:
# Cell 13: Re-export empire_viz_data.json from updated KG
# This exports the current KG state to JSON for the visualization.

import json

# Export all Colony nodes
nodes_result = run("""
    MATCH (n:Colony)
    WHERE n.established_year IS NOT NULL
    RETURN n.colony_id as id, 
           coalesce(n.canonical_name, n.name) as name,
           n.established_year as est, 
           n.independence_year as ind,
           n.region as region, 
           n.type as type, 
           n.wikidata_id as qid
    ORDER BY n.established_year, n.colony_id
""")

# Export all edges between Colony nodes
edges_result = run("""
    MATCH (s:Colony)-[r]->(t:Colony)
    WHERE type(r) IN ['EVOLVED_INTO', 'PARTITIONED_INTO', 'PART_OF', 'ADMINISTERED_UNDER']
    RETURN s.colony_id as source, t.colony_id as target, type(r) as type
    ORDER BY s.colony_id, t.id
""")

# Build export structure
export = {
    'nodes': [{k: v for k, v in node.items() if v is not None} for node in nodes_result],
    'edges': [dict(e) for e in edges_result],
}

# Write to file
output_path = '/home/jic823/textasdatacolonialofficelist/empire_viz_data.json'
with open(output_path, 'w') as f:
    json.dump(export, f, indent=None)  # compact format like original

print(f"Exported {len(export['nodes'])} nodes and {len(export['edges'])} edges")
print(f"Written to: {output_path}")
print(f"\nFile size: {len(json.dumps(export)):,} bytes")

# Summary by region
from collections import Counter
region_counts = Counter(n.get('region', 'Unknown') for n in export['nodes'])
print("\nNodes by region:")
for region, count in sorted(region_counts.items(), key=lambda x: -x[1]):
    print(f"  {region}: {count}")

In [22]:
# Cell 14: Historian review additions
# Addressing structural gaps identified during expert review.

# ── 14a: Princely States placeholder ──
# ~565 princely states were under British paramountcy (indirect rule).
# Adding a single placeholder node to represent them as a class.
# Paramountcy established post-Maratha Wars (~1818), lapsed at independence (1947).
# Most acceded to India; a few (Bahawalpur, Khairpur, etc.) to Pakistan.

run_write("""
    MERGE (n:Colony {colony_id: 'princely_states_placeholder_1818_1947'})
    ON CREATE SET
        n.canonical_name = 'Princely States (placeholder)',
        n.established_year = 1818,
        n.independence_year = 1947,
        n.region = 'South Asia',
        n.type = 'Princely State',
        n.wikidata_id = 'Q1336152'
""")
# Princely states → Dominion of India (most acceded)
run_write("""
    MATCH (ps:Colony {colony_id: 'princely_states_placeholder_1818_1947'}),
          (di:Colony {colony_id: 'dominion_of_india_1947_1950'})
    MERGE (ps)-[:EVOLVED_INTO]->(di)
""")
# Some princely states → Dominion of Pakistan
run_write("""
    MATCH (ps:Colony {colony_id: 'princely_states_placeholder_1818_1947'}),
          (dp:Colony {colony_id: 'dominion_of_pakistan_1947_1956'})
    MERGE (ps)-[:PARTITIONED_INTO]->(dp)
""")
print("14a: Princely States placeholder — created with edges to India + Pakistan")

# ── 14b: Pre-1831 South American colonies ──
# British Guiana (1831) was formed from three Dutch colonies seized during Napoleonic Wars:
# - Essequibo (Dutch 1616, British 1803)
# - Demerara (Dutch 1611, British 1803)
# - Berbice (Dutch 1627, British 1803)
# All three were formally ceded by Treaty of London (1814) and merged in 1831.

SA_NODES = [
    {
        'colony_id': 'essequibo_colony_1803_1831', 'canonical_name': 'Essequibo',
        'established_year': 1803, 'independence_year': 1831,
        'region': 'South America', 'type': 'Crown Colony', 'wikidata_id': 'Q2199929',
    },
    {
        'colony_id': 'demerara_colony_1803_1831', 'canonical_name': 'Demerara',
        'established_year': 1803, 'independence_year': 1831,
        'region': 'South America', 'type': 'Crown Colony', 'wikidata_id': 'Q2399707',
    },
    {
        'colony_id': 'berbice_colony_1803_1831', 'canonical_name': 'Berbice',
        'established_year': 1803, 'independence_year': 1831,
        'region': 'South America', 'type': 'Crown Colony', 'wikidata_id': 'Q861922',
    },
]

for node in SA_NODES:
    props = {k: v for k, v in node.items() if v is not None}
    set_clauses = ", ".join(f"n.{k} = ${k}" for k in props if k != 'colony_id')
    result = run_write(f"MERGE (n:Colony {{colony_id: $colony_id}}) ON CREATE SET {set_clauses}", **props)
    status = "CREATED" if "nodes_created" in result else "EXISTS"
    print(f"  {status}: {node['colony_id']}")

# All three → British Guiana (1831)
for src in ['essequibo_colony_1803_1831', 'demerara_colony_1803_1831', 'berbice_colony_1803_1831']:
    run_write("""
        MATCH (s:Colony {colony_id: $src}), (t:Colony {colony_id: 'british_guiana_1831_1966'})
        MERGE (s)-[:EVOLVED_INTO]->(t)
    """, src=src)

print("14b: Demerara, Essequibo, Berbice → British Guiana — created")

# ── 14c: Mosquito Coast and Bay Islands ──
CENTRAL_AM = [
    {
        'colony_id': 'mosquito_coast_1638_1860', 'canonical_name': 'Mosquito Coast',
        'established_year': 1638, 'independence_year': 1860,
        'region': 'Caribbean', 'type': 'Protectorate', 'wikidata_id': 'Q1948981',
    },
    {
        'colony_id': 'bay_islands_1852_1859', 'canonical_name': 'Bay Islands',
        'established_year': 1852, 'independence_year': 1859,
        'region': 'Caribbean', 'type': 'Crown Colony', 'wikidata_id': 'Q1553529',
    },
]

for node in CENTRAL_AM:
    props = {k: v for k, v in node.items() if v is not None}
    set_clauses = ", ".join(f"n.{k} = ${k}" for k in props if k != 'colony_id')
    run_write(f"MERGE (n:Colony {{colony_id: $colony_id}}) ON CREATE SET {set_clauses}", **props)
    print(f"  Created: {node['colony_id']}")

print("14c: Mosquito Coast + Bay Islands — created (standalone, no succession edges)")

# ── 14d: Dead-end successor nodes ──
# Territories that currently terminate without explaining how they left the Empire.

DEAD_END_NODES = [
    # Palestine → State of Israel (1948)
    {
        'colony_id': 'state_of_israel_1948', 'canonical_name': 'State of Israel',
        'established_year': 1948, 'independence_year': None,
        'region': 'Middle East', 'type': 'Independence', 'wikidata_id': 'Q801',
    },
    # British Somaliland → State of Somaliland (5 days!) → Somali Republic
    {
        'colony_id': 'somali_republic_1960', 'canonical_name': 'Somali Republic',
        'established_year': 1960, 'independence_year': None,
        'region': 'Middle East', 'type': 'Independence', 'wikidata_id': 'Q1045',
    },
]

for node in DEAD_END_NODES:
    props = {k: v for k, v in node.items() if v is not None}
    set_clauses = ", ".join(f"n.{k} = ${k}" for k in props if k != 'colony_id')
    run_write(f"MERGE (n:Colony {{colony_id: $colony_id}}) ON CREATE SET {set_clauses}", **props)

DEAD_END_EDGES = [
    # Palestine partitioned into Israel (+ Arab territories under Jordanian/Egyptian control)
    ('palestine_1920_1948', 'state_of_israel_1948', 'PARTITIONED_INTO', 'UN Partition / independence 1948'),
    # British Somaliland → Somali Republic (merged with Italian Somaliland)
    ('british_somaliland_1884_1960', 'somali_republic_1960', 'EVOLVED_INTO', 'independence + merger 1960'),
    # Ionian Islands → Greece (ceded 1864) — no successor node needed, just note the cession
    # Heligoland → Germany (ceded 1890) — terminal date already correct, no successor node needed
]

for src, tgt, etype, reason in DEAD_END_EDGES:
    result = run_write(f"""
        MATCH (s:Colony {{colony_id: $src}}), (t:Colony {{colony_id: $tgt}})
        MERGE (s)-[:{etype}]->(t)
    """, src=src, tgt=tgt)
    status = "OK" if "rels_created" in result else "EXISTS"
    print(f"  {status}: {src} -[{etype}]-> {tgt} — {reason}")

print("14d: Dead-end successors — Palestine→Israel, Somaliland→Somalia")

# ── 14e: West Indies Federation edges ──
# WIF (1958-1962) was a short-lived federation. Members didn't "evolve into" it —
# they joined while continuing to exist, then resumed independent paths when it dissolved.
# Modeling: members -[PART_OF]-> WIF (membership, not succession)
# WIF dissolution: members gained independence individually after 1962.

WIF_MEMBERS = [
    'jamaica_1655_1962',
    'trinidad_and_tobago_1889_1962',
    'barbados_1627_1966',
    'antigua_colony_1632_1981',
    'st_lucia_colony_1814_1979',
    'dominica_colony_1763_1978',
    'grenada_colony_1763_1974',
    'st_vincent_colony_1763_1979',
    'saint_christopher_nevis_anguilla_1958_1983',
    'montserrat_1632_ongoing',
]

for member_id in WIF_MEMBERS:
    result = run_write("""
        MATCH (m:Colony {colony_id: $mid}), (wif:Colony {colony_id: 'west_indies_federation_1958_1962'})
        MERGE (m)-[:PART_OF]->(wif)
    """, mid=member_id)
    if "rels_created" in result:
        print(f"  CREATED: {member_id} -[PART_OF]-> WIF")
    elif "no changes" in result:
        check = run("MATCH (n:Colony {colony_id: $id}) RETURN n.colony_id", id=member_id)
        if not check:
            print(f"  SKIP (not found): {member_id}")
        else:
            print(f"  EXISTS: {member_id} -[PART_OF]-> WIF")

print("14e: West Indies Federation membership edges — PART_OF (not succession)")

# ── Summary ──
print("\n=== Cell 14 Summary ===")
print("  Princely States placeholder (1818-1947) with edges to India + Pakistan")
print("  3 pre-British Guiana colonies (Essequibo, Demerara, Berbice)")
print("  2 Central American protectorates (Mosquito Coast, Bay Islands)")
print("  2 dead-end successor nodes (Israel, Somali Republic)")
print("  ~10 WIF membership edges (PART_OF)")

print("  Heligoland + Ionian Islands: terminal dates already correct, no successor nodes needed")

14a: Princely States placeholder — created with edges to India + Pakistan
  CREATED: essequibo_colony_1803_1831
  CREATED: demerara_colony_1803_1831
  CREATED: berbice_colony_1803_1831
14b: Demerara, Essequibo, Berbice → British Guiana — created
  Created: mosquito_coast_1638_1860
  Created: bay_islands_1852_1859
14c: Mosquito Coast + Bay Islands — created (standalone, no succession edges)
  OK: palestine_1920_1948 -[PARTITIONED_INTO]-> state_of_israel_1948 — UN Partition / independence 1948
  OK: british_somaliland_1884_1960 -[EVOLVED_INTO]-> somali_republic_1960 — independence + merger 1960
14d: Dead-end successors — Palestine→Israel, Somaliland→Somalia
  CREATED: jamaica_1655_1962 -[PART_OF]-> WIF
  CREATED: trinidad_and_tobago_1889_1962 -[PART_OF]-> WIF
  SKIP (not found): barbados_1627_1966
  CREATED: antigua_colony_1632_1981 -[PART_OF]-> WIF
  SKIP (not found): st_lucia_colony_1814_1979
  CREATED: dominica_colony_1763_1978 -[PART_OF]-> WIF
  CREATED: grenada_colony_1763_1974 -[PA